# KOSIS lexical vs BGE-M3 동일 조건 비교

기존 BGE 결과를 재사용하고 lexical Top-1·2·3·5만 같은 READY 39건, locked gold, Mapping-end 조건으로 실행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git'
BRANCH = 'codex/gold-v1-lock'
REPO_DIR = Path('/content/NLP_05-Team-Project-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_05-Team-Project-3')
INPUT_DIR = DRIVE_ROOT / 'inputs'
READY_CSV = INPUT_DIR / 'gold_measurement_scopefix_kosis_ready.csv'
GOLD_CSV = INPUT_DIR / 'gold_measurement_v1_locked.csv'
BGE_RUN_DIR = DRIVE_ROOT / 'runs' / 'bge_topk_mapping_end'
LEXICAL_RUN_DIR = DRIVE_ROOT / 'runs' / 'lexical_topk_mapping_end'
COMPARE_DIR = DRIVE_ROOT / 'runs' / 'lexical_vs_bge'
LEXICAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests', 'python-dotenv'], check=True)
print('commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
from google.colab import userdata

required = [
    READY_CSV,
    GOLD_CSV,
    BGE_RUN_DIR / 'topk_summary.csv',
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'필수 파일이 없습니다: {missing}')
if not os.environ.get('KOSIS_API_KEY'):
    os.environ['KOSIS_API_KEY'] = userdata.get('KOSIS_API_KEY')
if not os.environ.get('KOSIS_API_KEY'):
    raise RuntimeError('Colab 보안 비밀에 KOSIS_API_KEY를 등록하세요.')
print('입력, 기존 BGE 결과, API key 준비 완료')

In [ ]:
TABLE_INDEX = REPO_DIR / 'kosis_table_summary.csv'
stem = READY_CSV.stem
lexical_base_candidates = LEXICAL_RUN_DIR / 'base_top5' / f'{stem}_kosis_candidates_with_meta.csv'
lexical_base_meta = LEXICAL_RUN_DIR / 'base_top5' / f'{stem}_kosis_meta_index.csv'
reuse_lexical_base = lexical_base_candidates.exists() and lexical_base_meta.exists()
print('lexical 검색 결과 재사용:', reuse_lexical_base)

command = [
    sys.executable, str(REPO_DIR / 'run_kosis_topk_experiment.py'),
    '--input', str(READY_CSV),
    '--gold', str(GOLD_CSV),
    '--table-index', str(TABLE_INDEX),
    '--retrieval-mode', 'lexical',
    '--out-dir', str(LEXICAL_RUN_DIR),
    '--ks', '1', '2', '3', '5',
    '--delay', '0.05',
]
if reuse_lexical_base:
    command.append('--reuse-base')
subprocess.run(command, check=True)

In [ ]:
comparison_csv = COMPARE_DIR / 'lexical_vs_bge_topk_summary.csv'
comparison_report = COMPARE_DIR / 'lexical_vs_bge_topk_report.md'
subprocess.run([
    sys.executable, str(REPO_DIR / 'compare_kosis_topk_modes.py'),
    '--lexical-summary', str(LEXICAL_RUN_DIR / 'topk_summary.csv'),
    '--bge-summary', str(BGE_RUN_DIR / 'topk_summary.csv'),
    '--out-csv', str(comparison_csv),
    '--out-report', str(comparison_report),
], check=True)

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

display(pd.read_csv(comparison_csv, encoding='utf-8-sig'))
display(Markdown(comparison_report.read_text(encoding='utf-8')))

technical = pd.read_csv(
    LEXICAL_RUN_DIR / f'{READY_CSV.stem}_top5_technical_validated.csv',
    encoding='utf-8-sig',
)
print('\nlexical mapping_status')
print(technical['mapping_status'].value_counts(dropna=False))
print('\nlexical mapping_reason')
print(technical['mapping_reason'].value_counts(dropna=False).head(20))